In [ ]:
!pip install git+https://github.com/huggingface/transformers
!pip install pillow tqdm nltk rouge-score accelerate
!pip install bert-score

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-lk7xj89g
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-lk7xj89g
  Resolved https://github.com/huggingface/transformers to commit a30413b78feed68da5c486746f745db092bfdf9a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 12.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/2

In [ ]:
from huggingface_hub import login

# Login to Hugging Face
login(token="")

In [3]:
import nltk
try:
    nltk.data.find('tokenizers/punkt')
    print("NLTK punkt already downloaded")
except LookupError:
    print("Downloading NLTK punkt...")
    nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
import json
import os
import zipfile
from pathlib import Path
from transformers import AutoProcessor, Gemma3nForConditionalGeneration
import torch
import math
import gc
from PIL import Image
from tqdm import tqdm
from typing import List, Dict, Any

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import traceback
import requests
import os, json, math, gc, zipfile, traceback
from typing import Dict, Any, List
from tqdm import tqdm

from bert_score import score as bertscore_score

print("All imports successful!")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

All imports successful!
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA GeForce RTX 4090


In [8]:
# Clear entire HuggingFace cache (use with caution!)
!rm -rf /root/.cache/huggingface/hub/*

# Verify space
!df -h /root/.cache/huggingface/

Filesystem      Size  Used Avail Use% Mounted on
overlay          32G   12G   21G  37% /


In [ ]:
# Configuration
MODEL_NAME = "google/gemma-3n-e2b-it"

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print("Loading gemma-3-4b-it model and processor...")
print("This may take a few minutes on first run...")

model = Gemma3nForConditionalGeneration.from_pretrained(
    MODEL_NAME, device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("✓ Model loaded")

# Set devicea
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Using device: {device}")

print("\n=== Model Setup Complete ===")

Loading gemma-3-4b-it model and processor...
This may take a few minutes on first run...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✓ Model loaded
✓ Using device: cuda

=== Model Setup Complete ===


In [8]:
# Test with a simple example
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Okay, here's a detailed description of the image:

**Overall Impression:**

The image is a close-up shot of a vibrant garden scene, focusing on a cluster of pink cosmos flowers and a busy bumblebee. It has a slightly soft, natural feel, likely taken outdoors in daylight.

**Foreground:**

*   **Cosmos Flowers:** The dominant feature is a large, bright pink cosmos flower in the center. It’s fully open, displaying its layered petals and a


In [ ]:
class VLMBenchmark:
    def __init__(
        self,
        processor,
        model,
        device,
        checkpoint_file: str = "checkpoint.json",
        bertscore_model_type: str = "bert-base-multilingual-cased",
        bertscore_lang: str = "si",
        bertscore_batch_size: int = 64,
    ):
        """Initialize the benchmark with existing model and processor."""
        print("Initializing benchmark...")

        self.processor = processor
        self.model = model
        self.device = device
        self.checkpoint_file = checkpoint_file

        # BERTScore config (computed at the very end)
        self.bertscore_model_type = bertscore_model_type
        self.bertscore_lang = bertscore_lang
        self.bertscore_batch_size = bertscore_batch_size

        self.results = []
        self.errors = []
        self.processed_ids = set()

        print("✓ Benchmark initialized")

    # -----------------------------
    # Checkpointing
    # -----------------------------
    def load_checkpoint(self):
        """Load checkpoint if it exists."""
        if os.path.exists(self.checkpoint_file):
            print(f"Loading checkpoint from {self.checkpoint_file}...")
            try:
                with open(self.checkpoint_file, "r", encoding="utf-8") as f:
                    checkpoint = json.load(f)
                self.results = checkpoint.get("results", [])
                self.errors = checkpoint.get("errors", [])
                self.processed_ids = set(checkpoint.get("processed_ids", []))
                print(f"✓ Resumed: {len(self.results)} items, {len(self.errors)} errors")
                return True
            except Exception as e:
                print(f"Error loading checkpoint: {e}")
                return False
        return False

    def save_checkpoint(self):
        """Save checkpoint after each image."""
        try:
            checkpoint = {
                "results": self.results,
                "errors": self.errors,
                "processed_ids": list(self.processed_ids),
            }
            with open(self.checkpoint_file, "w", encoding="utf-8") as f:
                json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"Error saving checkpoint: {e}")

    # -----------------------------
    # IO helpers
    # -----------------------------
    def unzip_images(self, zip_path: str, extract_to: str):
        """Unzip the image dataset."""
        print(f"Unzipping {zip_path} to {extract_to}...")

        if os.path.exists(extract_to):
            print(f"✓ Dataset folder already exists at {extract_to}")
            return

        try:
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_to)
            print(f"✓ Successfully extracted to {extract_to}")
        except Exception as e:
            print(f"Error unzipping: {e}")
            raise

    def load_json(self, json_path: str) -> List[Dict]:
        """Load the input JSON file."""
        print(f"Loading JSON from {json_path}...")
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✓ Loaded {len(data)} items")
        return data

    def find_image(self, image_id: int, dataset_folder: str) -> str:
        """Find image file by ID in the dataset folder."""
        extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

        for ext in extensions:
            image_path = os.path.join(dataset_folder, f"{image_id}{ext}")
            if os.path.exists(image_path):
                return image_path

            for root, _, _files in os.walk(dataset_folder):
                image_path = os.path.join(root, f"{image_id}{ext}")
                if os.path.exists(image_path):
                    return image_path

        raise FileNotFoundError(f"Image with ID {image_id} not found in {dataset_folder}")

    # -----------------------------
    # Metrics (BLEU only during run)
    # -----------------------------
    def calculate_bleu(self, reference: str, hypothesis: str) -> float:
        """Calculate BLEU score (character-level)."""
        try:
            reference_tokens = list(reference.strip())
            hypothesis_tokens = list(hypothesis.strip())
            smoothing = SmoothingFunction().method1
            bleu_score = sentence_bleu(
                [reference_tokens],
                hypothesis_tokens,
                smoothing_function=smoothing,
            )
            return bleu_score * 100
        except Exception as e:
            print(f"Error calculating BLEU: {e}")
            return 0.0

    # -----------------------------
    # Core QA processing
    # -----------------------------
    def process_single_qa(
        self,
        image_path: str,
        question: str,
        ground_truth: str,
        qa_id: int,
        image_id: int,
    ) -> Dict[str, Any]:
        """Process a single question-answer pair."""
        try:
            messages = [
                
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path},
                        {"type": "text", "text": f"මෙම රූපය බලා පහත ප්‍රශ්නයට පිළිතුරු දෙන්න:\nප්‍රශ්නය: {question}"},
                    ],
                },
            ]

            inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(self.model.device, dtype=torch.bfloat16)

            input_len = inputs["input_ids"].shape[-1]

            with torch.inference_mode():
                generated_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=300,
                    do_sample=False,
                )

            generated_ids_trimmed = generated_ids[0][input_len:]
            generated_text = self.processor.decode(generated_ids_trimmed, skip_special_tokens=True)

            # Perplexity / loss on generated tokens
            loss = None
            perplexity = None
            try:
                attention_mask = torch.ones_like(generated_ids)
                labels = generated_ids.clone()
                labels[:, :input_len] = -100

                with torch.inference_mode():
                    outputs = self.model(
                        input_ids=generated_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                    )

                if outputs.loss is not None:
                    loss = outputs.loss.item()
                    perplexity = math.exp(loss)
                else:
                    print(f"Warning: Loss is None for QA {qa_id}")

            except Exception as e:
                print(f"Error calculating perplexity for QA {qa_id}: {type(e).__name__}: {str(e)}")
                print(traceback.format_exc())

            bleu_score = self.calculate_bleu(ground_truth, generated_text)

            # NOTE: BERTScore is computed later in one pass (end of run)
            return {
                "answer": generated_text,
                "ground_truth": ground_truth,
                "qa_id": qa_id,
                "perplexity": perplexity,
                "loss": loss,
                "bleu_score": bleu_score,
                "bertscore": None,  # filled later
                "status": "success",
            }

        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"Error processing QA {qa_id}: {error_msg}")
            print(f"Traceback:\n{traceback.format_exc()}")

            self.errors.append(
                {
                    "qa_id": qa_id,
                    "id": image_id,
                    "error": error_msg,
                    "traceback": traceback.format_exc(),
                }
            )
            return {
                "answer": None,
                "ground_truth": ground_truth,
                "qa_id": qa_id,
                "perplexity": None,
                "loss": None,
                "bleu_score": None,
                "bertscore": None,
                "status": "error",
                "error": error_msg,
            }

    def process_dataset(self, json_path: str, dataset_folder: str, save_every: int = 10):
        """Process the entire dataset (generation only; no BERTScore here)."""
        data = self.load_json(json_path)

        items_to_process = [item for item in data if item["id"] not in self.processed_ids]
        if len(items_to_process) < len(data):
            print(f"Skipping {len(data) - len(items_to_process)} already processed items")

        for idx, item in enumerate(tqdm(items_to_process, desc="Processing items")):
            image_id = item["id"]

            try:
                image_path = self.find_image(image_id, dataset_folder)

                result_qas = []
                for qa in tqdm(item["qas"], desc=f"Image {image_id}", leave=False):
                    qa_result = self.process_single_qa(
                        image_path=image_path,
                        question=qa["question"],
                        ground_truth=qa["answer"],
                        qa_id=qa["qa_id"],
                        image_id=image_id,
                    )
                    result_qas.append(qa_result)

                    torch.cuda.empty_cache()
                    gc.collect()

                self.results.append({"id": image_id, "qas": result_qas})
                self.processed_ids.add(image_id)

                if (idx + 1) % save_every == 0:
                    self.save_checkpoint()

            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)}"
                print(f"Error processing image {image_id}: {error_msg}")
                for qa in item["qas"]:
                    self.errors.append({"qa_id": qa["qa_id"], "id": image_id, "error": error_msg})
                self.processed_ids.add(image_id)
                self.save_checkpoint()

        self.save_checkpoint()

    # -----------------------------
    # BERTScore (computed at end)
    # -----------------------------
    def run_bertscore(self):
        """
        Compute BERTScore for all successful QAs once, at the end.
        Writes per-QA bertscore: {"precision":..., "recall":..., "f1":...} (0-100).
        """
        print("\nComputing BERTScore for all successful QAs (end-of-run)...")

        # Collect (pred, ref) with pointers back to result objects
        preds: List[str] = []
        refs: List[str] = []
        pointers: List[Dict[str, Any]] = []

        for item in self.results:
            for qa in item["qas"]:
                if qa.get("status") == "success" and qa.get("answer") is not None and qa.get("ground_truth") is not None:
                    preds.append(qa["answer"])
                    refs.append(qa["ground_truth"])
                    pointers.append(qa)

        if not preds:
            print("No successful samples found for BERTScore.")
            return

        try:
            P, R, F1 = bertscore_score(
                cands=preds,
                refs=refs,
                lang=self.bertscore_lang,
                model_type=self.bertscore_model_type,
                verbose=True,
                batch_size=self.bertscore_batch_size,
                device=str(self.device) if isinstance(self.device, str) else None,
            )

            # Write back to each QA (store as percent)
            for qa, p, r, f in zip(pointers, P, R, F1):
                qa["bertscore"] = {
                    "precision": float(p.item() * 100.0),
                    "recall": float(r.item() * 100.0),
                    "f1": float(f.item() * 100.0),
                }

            print(f"✓ BERTScore computed for {len(pointers)} samples")

        except Exception as e:
            print(f"Error computing BERTScore: {type(e).__name__}: {str(e)}")
            print(traceback.format_exc())

    # -----------------------------
    # Aggregates & saving
    # -----------------------------
    def calculate_aggregate_metrics(self) -> Dict[str, float]:
        """Calculate aggregate metrics (BLEU, BERTScore, perplexity, loss)."""
        all_bleu = []
        all_perplexity = []
        all_loss = []
        all_bs_p, all_bs_r, all_bs_f1 = [], [], []

        total_qas = 0
        success_qas = 0

        for item in self.results:
            for qa in item["qas"]:
                total_qas += 1
                if qa.get("status") != "success":
                    continue

                success_qas += 1

                if qa.get("bleu_score") is not None:
                    all_bleu.append(qa["bleu_score"])
                if qa.get("perplexity") is not None:
                    all_perplexity.append(qa["perplexity"])
                if qa.get("loss") is not None:
                    all_loss.append(qa["loss"])

                bs = qa.get("bertscore")
                if bs:
                    all_bs_p.append(bs["precision"])
                    all_bs_r.append(bs["recall"])
                    all_bs_f1.append(bs["f1"])

        def mean(xs: List[float]) -> float:
            return (sum(xs) / len(xs)) if xs else 0.0

        return {
            "mean_bleu": mean(all_bleu),
            "mean_bertscore_precision": mean(all_bs_p),
            "mean_bertscore_recall": mean(all_bs_r),
            "mean_bertscore_f1": mean(all_bs_f1),
            "mean_perplexity": mean(all_perplexity),
            "mean_loss": mean(all_loss),
            "total_samples": success_qas,
            "success_rate": (success_qas / total_qas * 100.0) if total_qas else 0.0,
        }

    def save_results(self, output_path: str):
        """Save results to JSON file."""
        print(f"Saving results to {output_path}...")

        aggregate_metrics = self.calculate_aggregate_metrics()
        output_data = {
            "results": self.results,
            "aggregate_metrics": aggregate_metrics,
        }

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

        print(f"✓ Saved {len(self.results)} results")
        print("\n=== Aggregate Metrics ===")
        print(f"Mean BLEU Score: {aggregate_metrics['mean_bleu']:.2f}")
        print(f"Mean BERTScore P: {aggregate_metrics['mean_bertscore_precision']:.2f}")
        print(f"Mean BERTScore R: {aggregate_metrics['mean_bertscore_recall']:.2f}")
        print(f"Mean BERTScore F1: {aggregate_metrics['mean_bertscore_f1']:.2f}")
        print(f"Mean Perplexity: {aggregate_metrics['mean_perplexity']:.4f}")
        print(f"Mean Loss: {aggregate_metrics['mean_loss']:.4f}")
        print(f"Success Rate: {aggregate_metrics['success_rate']:.2f}%")

    def save_errors(self, error_path: str):
        """Save errors to JSON file."""
        if self.errors:
            print(f"Saving {len(self.errors)} errors to {error_path}...")
            with open(error_path, "w", encoding="utf-8") as f:
                json.dump(self.errors, f, ensure_ascii=False, indent=2)
        else:
            print("No errors to save!")


print("✓ Gemma Benchmark class defined")


✓ Gemma Benchmark class defined


In [ ]:
# -----------------------------
# Usage
# -----------------------------
# Configuration
INPUT_JSON = "../data/40k-qas-test.json"
ZIP_FILE = "../data/filtered_images.zip"
DATASET_FOLDER = "../data/dataset"
OUTPUT_JSON = "benchmarking-results-gemma-3n-e2b.json"
ERROR_JSON = "errors.json"
CHECKPOINT_FILE = "checkpoint-gemma-3n-e2b.json"
SAVE_EVERY = 5

# Initialize benchmark
benchmark = VLMBenchmark(
    processor,
    model,
    device,
    checkpoint_file=CHECKPOINT_FILE,
    bertscore_model_type="bert-base-multilingual-cased",
    bertscore_lang="si",
    bertscore_batch_size=64,
)

# Load checkpoint if exists
benchmark.load_checkpoint()

# Unzip images if needed
if not os.path.exists(DATASET_FOLDER):
    benchmark.unzip_images(ZIP_FILE, DATASET_FOLDER)

# 1) Generate answers + compute BLEU/perplexity/loss
benchmark.process_dataset(INPUT_JSON, DATASET_FOLDER, save_every=SAVE_EVERY)

# 2) Compute BERTScore once at the end
benchmark.run_bertscore()

# 3) Save final results
benchmark.save_results(OUTPUT_JSON)
benchmark.save_errors(ERROR_JSON)

print("\n=== Benchmark Complete ===")
print(f"Total items processed: {len(benchmark.results)}")
print(f"Total errors: {len(benchmark.errors)}")

Initializing benchmark...
✓ Benchmark initialized
Loading JSON from ../data/40k-qas-test.json...
✓ Loaded 2731 items


Image 2411192:   0%|          | 0/2 [00:00<?, ?it/s]